In [ ]:
# Mount Google Drive to access input data
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# ============================================================
# Imports, Configuration, and Data Load
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

# --- User configuration ---
DATA_PATH = "/content/drive/MyDrive/<YOUR_PROJECT_FOLDER>/NOAA CRW/Data/bleaching_confusion_labeled.csv"

# MPAs retained for the two-site comparison figures (Figures 4 and S2 facet subset)
TARGET_MPAS = ['Turneffe Atoll Marine Reserve', 'Port Honduras Marine Reserve']

# Sensor columns and axis labels used across both figures
SENSORS = {
    'CRW':       ('CRW_SST_Monthly_Max',       'CRW Monthly Max SST (°C)'),
    'ECOSTRESS': ('ECOSTRESS_LST_Monthly_Max',  'ECOSTRESS Monthly Max LST (°C)'),
}

# MPA color palette (consistent across figures)
PALETTE = {
    'Turneffe Atoll Marine Reserve': '#1f78b4',
    'Port Honduras Marine Reserve':  '#e31a1c',
}

# --- Load and prepare data ---
df = pd.read_csv(DATA_PATH)

# Subset to the two focal MPAs used in Figures 4 and S2
df_sub = df[df['Marine Protected Area Name'].isin(TARGET_MPAS)].copy()


In [ ]:
# ============================================================
# Figure S2: Coral Bleaching vs ECOSTRESS LST by MPA (All Sites)
# ============================================================
# Faceted scatter plot with linear regression lines showing the
# relationship between ECOSTRESS monthly max LST and percent
# coral bleaching across all MPAs in the dataset.
#
# Bleaching values are clipped to [0, 100] before plotting.
# Shared axis limits are enforced across panels for comparability.
# R² and p-value annotations are placed in the lower-right of
# each panel; panels with fewer than 3 observations are labelled
# 'n < 3' and no regression is fitted.
# ============================================================

# Clip any out-of-range bleaching values to [0, 100]
df_plot = df.copy()
df_plot['Bleached_Percent'] = df_plot['Bleached_Percent'].clip(lower=0, upper=100)

g = sns.lmplot(
    data=df_plot,
    x='ECOSTRESS_LST_Monthly_Max',
    y='Bleached_Percent',
    col='Marine Protected Area Name',
    hue='Marine Protected Area Name',
    palette='Paired',
    col_wrap=4,
    height=3,
    sharex=True,
    sharey=True,
    legend=False
)

# Enforce consistent axis limits across all panels
x_min = df_plot['ECOSTRESS_LST_Monthly_Max'].min()
x_max = df_plot['ECOSTRESS_LST_Monthly_Max'].max()
x_pad = (x_max - x_min) * 0.05
for ax in g.axes.flat:
    ax.set_xlim(x_min - x_pad, x_max + x_pad)
    ax.set_ylim(0, 100)

g.set_titles("{col_name}", size=10)

# Annotate each panel with R² and p-value (lower right)
for ax in g.axes.flat:
    mpa_name = ax.get_title()
    if not mpa_name:
        continue
    subset = df_plot[df_plot['Marine Protected Area Name'] == mpa_name]
    subset = subset.dropna(subset=['ECOSTRESS_LST_Monthly_Max', 'Bleached_Percent'])

    if len(subset) > 2:
        r, p = pearsonr(subset['ECOSTRESS_LST_Monthly_Max'], subset['Bleached_Percent'])
        stat_text = f'$R^2$ = {r**2:.2f}, p = {p:.3f}' if np.isfinite(r) else 'Calculation Error'
    else:
        stat_text = 'n < 3'

    ax.text(0.95, 0.05, stat_text, transform=ax.transAxes,
            fontsize=9, ha='right', va='bottom')

g.fig.suptitle('Coral Bleaching vs ECOSTRESS LST by MPA', y=1.03)
g.set_axis_labels("ECOSTRESS Monthly Max LST (°C)", "Bleached Corals (%)")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Figure 4: Bleaching vs LST and SST for Two Focal MPAs
# ============================================================
# 2×2 panel figure comparing the relationship between coral
# bleaching and temperature for two sensors (CRW SST, ECOSTRESS
# LST) across two focal MPAs (Turneffe Atoll, Port Honduras).
#
# Layout:
#   Rows    = MPAs  (Turneffe top, Port Honduras bottom)
#   Columns = Sensors (CRW left, ECOSTRESS right)
#
# Each panel shows a scatter plot with a linear regression line,
# an R²/p/n annotation in the lower right, and an A–D panel label
# in the upper left. MPA names are annotated on the right margin.
# ============================================================

panel_labels = [['A', 'B'], ['C', 'D']]

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(9, 7), sharey=True)

for row_idx, mpa in enumerate(TARGET_MPAS):
    for col_idx, (sensor_label, (x_col, x_axis_label)) in enumerate(SENSORS.items()):
        ax     = axes[row_idx, col_idx]
        color  = PALETTE[mpa]
        subset = df_sub[df_sub['Marine Protected Area Name'] == mpa].dropna(
            subset=[x_col, 'Bleached_Percent']
        )
        n = len(subset)

        if n > 2:
            sns.regplot(
                data=subset, x=x_col, y='Bleached_Percent', ax=ax,
                color=color,
                scatter_kws={'alpha': 0.6, 's': 40},
                line_kws={'linewidth': 1.5}
            )
            r, p   = pearsonr(subset[x_col], subset['Bleached_Percent'])
            p_str  = f'p = {p:.3f}' if p >= 0.001 else 'p < 0.001'
            stat_text = f'$R^2$ = {r**2:.2f}, {p_str}\nn = {n}'
        else:
            ax.scatter(subset[x_col], subset['Bleached_Percent'],
                       color=color, alpha=0.6, s=40)
            stat_text = 'n < 3'

        # Stats annotation — bottom right
        ax.text(0.95, 0.05, stat_text, transform=ax.transAxes,
                fontsize=9, va='bottom', ha='right',
                bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.7))

        # Panel letter — top left
        ax.text(0.03, 0.97, panel_labels[row_idx][col_idx], transform=ax.transAxes,
                fontsize=12, fontweight='bold', va='top', ha='left')

        # Sensor column titles on top row only
        if row_idx == 0:
            ax.set_title(sensor_label, fontsize=11, fontweight='bold')

        # MPA row label on right margin
        if col_idx == 1:
            ax.annotate(mpa, xy=(1.02, 0.5), xycoords='axes fraction',
                        fontsize=9, fontweight='bold', va='center', rotation=270)

        ax.set_xlabel(x_axis_label, fontsize=9)
        ax.set_ylabel('Bleached Corals (%)' if col_idx == 0 else '', fontsize=9)

plt.tight_layout()
plt.show()
